# 03 — Integration, Feature Engineering & Encoding

**Purpose:** Merge all cleaned tables into a master table, build economic features, and encode categoricals.  
**Input:** `data/processed/cleaned/*.csv` (9 clean tables)  
**Output:** `data/processed/master_table.csv` — one large integrated table ready for final filtering

---

## Section Overview

1. **Build intermediate tables** — pivot FAO/World Bank sources into one row per (product × year) or (country) format  
2. **Build master table** — left-join all intermediates onto `world_imports` as the base  
3. **Add Algeria exports** — attach the target variable  
4. **Engineer features** — export gap, market share, price competitiveness, demand growth, log transforms  
5. **Encode categoricals** — one-hot encode product sector

## Setup

In [ ]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.4f}'.format)

CLEAN_DIR = Path('../data/processed/cleaned')
PROC_DIR  = Path('../data/processed')

YEAR_MIN, YEAR_MAX = 2018, 2022

## 1. Load Cleaned Tables

In [ ]:
tables = [
    'world_imports', 'algeria_production', 'algeria_inputs', 'algeria_land',
    'algeria_prices', 'algeria_worldbank', 'unit_values', 'country_metadata',
    'algeria_exports',
]

dfs = {}
for name in tables:
    dfs[name] = pd.read_csv(CLEAN_DIR / f'{name}_clean.csv', low_memory=False)
    print(f"✅  {name:<22} {len(dfs[name]):>8,} rows  {len(dfs[name].columns):>3} cols")

---
## Step 1 — Build `trade_core`

Append World-average unit value rows to `world_imports`.

**Why:** `unit_values` contains world-average USD/kg prices keyed by (hs_code_6digit, year, exporter_name).  
The `World` rows from `unit_values` represent the benchmark price Algeria competes against.  
We attach these directly to `world_imports` via a left-join on (hs_code_6digit, year).

In [ ]:
# Extract world-average unit values (one row per product × year)
world_unit_values = (
    dfs['unit_values']
    [dfs['unit_values']['exporter_name'] == 'World']
    [['hs_code_6digit', 'year', 'unit_value_usd_per_kg']]
    .drop_duplicates(subset=['hs_code_6digit', 'year'])
)

trade_core = dfs['world_imports'].merge(
    world_unit_values,
    on=['hs_code_6digit', 'year'],
    how='left',
    suffixes=('', '_world_avg'),
)

# Remove Comtrade aggregate region (not a real country)
trade_core = trade_core[trade_core['importer_iso3'] != 'S19']

print(f"trade_core shape: {trade_core.shape}")
print(f"World exporter rows: {(trade_core['exporter_name'] == 'World').sum():,}")
print(f"Bilateral rows     : {(trade_core['exporter_name'] != 'World').sum():,}")
print(f"unit_value_usd_per_kg null: {trade_core['unit_value_usd_per_kg'].isnull().sum():,}")

---
## Step 2 — Build `algeria_product_yearly`

Pivot FAO production data → one row per (hs_code_6digit × year).

In [ ]:
prod = dfs['algeria_production'].copy()
prod = prod[prod['Year'].between(YEAR_MIN, YEAR_MAX)]

algeria_product_yearly = prod.pivot_table(
    index=['hs_code_6digit', 'Item', 'Year'],
    columns='Element',
    values='Value',
    aggfunc='mean',
).reset_index()

algeria_product_yearly.columns.name = None

# Clean column names
def clean_col(c):
    if c in ('hs_code_6digit', 'Item', 'Year'):
        return c
    return 'prod_' + c.lower().replace(' ', '_').replace('/', '_')

algeria_product_yearly.columns = [clean_col(c) for c in algeria_product_yearly.columns]
algeria_product_yearly = algeria_product_yearly.rename(columns={'Year': 'year'})

print(f"algeria_product_yearly shape: {algeria_product_yearly.shape}")
print(f"Columns: {algeria_product_yearly.columns.tolist()}")

---
## Step 3 — Build `algeria_yearly_context`

Aggregate prices, inputs, land data → one row per year (Algeria macro context).

In [ ]:
# --- Prices: producer price index & USD/tonne ---
prices_pivot = (
    dfs['algeria_prices']
    .groupby(['Year', 'Element'])['Value']
    .mean()
    .unstack()
)

def price_colname(c):
    return 'price_' + c.lower().replace(' ', '_').replace('(', '').replace(')', '').replace('-', '_').replace('/', '_')[:30]

prices_pivot.columns = [price_colname(c) for c in prices_pivot.columns]
prices_pivot = prices_pivot.reset_index().rename(columns={'Year': 'year'})

# Forward-fill 2022 if USD price is missing (FAOSTAT often lags one year)
if 'price_producer_price_usd_tonne' in prices_pivot.columns:
    prices_pivot['price_producer_price_usd_tonne'] = prices_pivot['price_producer_price_usd_tonne'].ffill()

# --- Inputs: fertilizer agricultural use + import quantity ---
inputs_pivot = (
    dfs['algeria_inputs']
    .groupby(['Year', 'Item', 'Element'])['Value']
    .sum()
    .unstack()
)
inputs_pivot.columns = [f'fertilizer_{c.lower().replace(" ","_")}' for c in inputs_pivot.columns]
inputs_pivot = inputs_pivot.reset_index().rename(columns={'Year': 'year'})
inputs_pivot = inputs_pivot.groupby('year')[list(inputs_pivot.filter(like='fertilizer').columns)].sum().reset_index()

# --- Land: area, share, productivity ---
land_pivot = (
    dfs['algeria_land']
    .groupby(['Year', 'Item', 'Element'])['Value']
    .mean()
    .unstack()
)
land_cols = [
    'land_' + c.lower().replace(' ', '_').replace('(', '').replace(')', '').replace('.', '').replace('$', 'usd')[:35]
    for c in land_pivot.columns
]
land_pivot.columns = land_cols
land_pivot = land_pivot.reset_index().rename(columns={'Year': 'year'})
land_pivot = land_pivot.groupby('year').mean(numeric_only=True).reset_index()

# --- Merge all into yearly context ---
algeria_yearly_context = (
    prices_pivot
    .merge(inputs_pivot, on='year', how='outer')
    .merge(land_pivot,   on='year', how='outer')
)
algeria_yearly_context = algeria_yearly_context[algeria_yearly_context['year'].between(YEAR_MIN, YEAR_MAX)]

print(f"algeria_yearly_context shape: {algeria_yearly_context.shape}")
print(algeria_yearly_context)

---
## Step 4 — Build `importer_profile`

Pivot country_metadata → one row per country.

In [ ]:
importer_profile = dfs['country_metadata'].pivot_table(
    index=['country_iso3', 'country_name'],
    columns='indicator_label',
    values='value',
    aggfunc='first',
).reset_index()

importer_profile.columns.name = None
importer_profile = importer_profile.rename(columns={'country_iso3': 'importer_iso3'})

print(f"importer_profile shape: {importer_profile.shape}")
print(f"Columns: {importer_profile.columns.tolist()}")

---
## Step 5 — Build Master Table

Left-join everything onto `trade_core`.

In [ ]:
master = trade_core.copy()

master = master.merge(algeria_product_yearly, on=['hs_code_6digit', 'year'], how='left')
master = master.merge(algeria_yearly_context, on='year',                     how='left')
master = master.merge(importer_profile,       on='importer_iso3',            how='left')

print(f"Master after joins: {master.shape}")

---
## Step 6 — Attach Algeria Exports (Target Variable)

Merge Algeria's actual export values → this becomes the target.  
Missing values mean Algeria does not export that product to that country → fill with 0.

In [ ]:
algeria_exp = (
    dfs['algeria_exports']
    [['hs_code_6digit', 'partner_name', 'year', 'trade_value_usd', 'unit_value_usd_per_kg']]
    .copy()
    .rename(columns={
        'trade_value_usd'      : 'algeria_export_value_usd',
        'unit_value_usd_per_kg': 'algeria_export_unit_value',
        'partner_name'         : 'importer_name',
    })
)

master = master.merge(
    algeria_exp,
    on=['hs_code_6digit', 'importer_name', 'year'],
    how='left',
)

# NaN = no export recorded → 0
master['algeria_export_value_usd']  = master['algeria_export_value_usd'].fillna(0)
master['algeria_export_unit_value'] = master['algeria_export_unit_value'].fillna(0)

print(f"Algeria exports > 0 : {(master['algeria_export_value_usd'] > 0).sum():,}  (existing markets)")
print(f"Algeria exports = 0 : {(master['algeria_export_value_usd'] == 0).sum():,}  (untapped markets)")

---
## Step 7 — Scope Flags

Classify each row by its usefulness for opportunity modeling.  

- `is_resource_available`: Algeria has this product (expert-flagged)  
- `is_world_demand_row`: `exporter_name = 'World'` → represents total market demand  
- `in_model_scope`: both flags True → the rows we actually model

In [ ]:
resource_col = 'algeria_has_resource' if 'algeria_has_resource' in master.columns else 'is_resource_available'

master['is_resource_available'] = master[resource_col].astype(bool)
master['is_world_demand_row']   = (master['exporter_name'] == 'World')
master['in_model_scope']        = master['is_resource_available'] & master['is_world_demand_row']

print(f"in_model_scope = True  : {master['in_model_scope'].sum():,} rows")
print(f"in_model_scope = False : {(~master['in_model_scope']).sum():,} rows")

---
## Step 8 — Feature Engineering

Build economic signals from the joined table.

**All features derived from existing columns — no external data introduced.**

| Feature | Formula | Meaningful on |
|---|---|---|
| `export_gap_usd` | `trade_value_usd - algeria_export_value_usd` | World rows only |
| `algeria_market_share` | `algeria_export_value_usd / trade_value_usd` | World rows only |
| `price_competitiveness` | `algeria_export_unit_value / unit_value_usd_per_kg` | World rows only |
| `demand_growth_yoy` | YoY % change in `trade_value_usd` | All rows |
| `production_capacity_ratio` | `prod_production / trade_value_usd` | World rows only |
| `log_trade_value` | `log(1 + trade_value_usd)` | All rows |
| `log_gdp_per_capita` | `log(1 + gdp_per_capita_usd)` | All rows |
| `log_population` | `log(1 + population_total)` | All rows |

In [ ]:
bilateral_mask = ~master['is_world_demand_row']  # True for bilateral rows

# ── Export Gap ─────────────────────────────────────────────────────────────
master['export_gap_usd'] = master['trade_value_usd'] - master['algeria_export_value_usd']
master.loc[bilateral_mask, 'export_gap_usd'] = np.nan

# ── Algeria Market Share ───────────────────────────────────────────────────
master['algeria_market_share'] = (
    master['algeria_export_value_usd'] / master['trade_value_usd']
).replace([np.inf, -np.inf], np.nan)
master.loc[bilateral_mask, 'algeria_market_share'] = np.nan
assert (master['algeria_market_share'].dropna() > 1).sum() == 0, "Market share > 1 detected"

# ── Price Competitiveness ──────────────────────────────────────────────────
master['price_competitiveness'] = (
    master['algeria_export_unit_value'] / master['unit_value_usd_per_kg']
).replace([np.inf, -np.inf], np.nan)
master.loc[bilateral_mask, 'price_competitiveness'] = np.nan
# Cap extreme outliers (ratio > 10x world avg is likely data error)
master.loc[master['price_competitiveness'] > 10, 'price_competitiveness'] = np.nan

# ── Demand Growth YoY ─────────────────────────────────────────────────────
master = master.sort_values(['hs_code_6digit', 'importer_iso3', 'exporter_name', 'year'])
master['demand_growth_yoy'] = (
    master
    .groupby(['hs_code_6digit', 'importer_iso3', 'exporter_name'])['trade_value_usd']
    .pct_change()
    .clip(-1, 10)  # cap at -100% to +1000%
)

# ── Production Capacity Ratio ──────────────────────────────────────────────
master['production_capacity_ratio'] = (
    master['prod_production'] / master['trade_value_usd']
).replace([np.inf, -np.inf], np.nan)
master.loc[bilateral_mask, 'production_capacity_ratio'] = np.nan
p99_pcr = master['production_capacity_ratio'].quantile(0.99)
master['production_capacity_ratio'] = master['production_capacity_ratio'].clip(upper=p99_pcr)

# ── Log Transforms ─────────────────────────────────────────────────────────
master['log_trade_value']    = np.log1p(master['trade_value_usd'])

gdp_pc_col  = 'gdp_per_capita_usd'  if 'gdp_per_capita_usd'  in master.columns else None
pop_col     = 'population_total'     if 'population_total'     in master.columns else None

if gdp_pc_col:
    master['log_gdp_per_capita'] = np.log1p(master[gdp_pc_col])
if pop_col:
    master['log_population'] = np.log1p(master[pop_col])

print("Feature engineering complete")
print(f"New columns added: export_gap_usd, algeria_market_share, price_competitiveness,")
print(f"                   demand_growth_yoy, production_capacity_ratio,")
print(f"                   log_trade_value, log_gdp_per_capita, log_population")

In [ ]:
# Sanity checks on feature ranges
world_rows = master[master['is_world_demand_row']]

print("export_gap_usd (World rows only):")
print(world_rows['export_gap_usd'].describe(), "\n")

print("algeria_market_share (World rows only):")
print(world_rows['algeria_market_share'].describe(), "\n")

print("price_competitiveness (World rows only):")
print(world_rows['price_competitiveness'].describe(), "\n")

print("demand_growth_yoy (all rows):")
print(master['demand_growth_yoy'].describe())

---
## Step 9 — Encode Product Sector (One-Hot)

Product sector has ~14 categories — small enough for one-hot encoding.  
Chapter dummies are **not** created: sector already captures this at the right granularity.

In [ ]:
sector_col = 'product_sector' if 'product_sector' in master.columns else None

if sector_col:
    sector_dummies = pd.get_dummies(master[sector_col], prefix='sector')
    master = pd.concat([master, sector_dummies], axis=1)
    print(f"Sector dummies added: {sector_dummies.columns.tolist()}")
else:
    print("⚠️  product_sector column not found — verify column name in world_imports")

# Note on importer_iso3 target encoding:
# This is deliberately EXCLUDED here.
# Target encoding (using mean of the label per country) must be computed
# ONLY on training data after the train/test split to avoid data leakage.
# See 04_final_preparation.ipynb handoff notes.
print("\nNote: importer_iso3 target encoding intentionally deferred to modeling step")
print("      (must be fit on train set only to avoid temporal leakage)")

---
## Step 10 — Save Master Table

In [ ]:
out_path = PROC_DIR / 'master_table.csv'
master.to_csv(out_path, index=False)

print(f"master_table.csv saved ✅")
print(f"  Shape         : {master.shape[0]:,} rows × {master.shape[1]} cols")
print(f"  In model scope: {master['in_model_scope'].sum():,} rows")
print(f"  Path          : {out_path.resolve()}")
print(f"\nColumn count by type:")
print(master.dtypes.value_counts())

---
**Next:** `04_final_preparation.ipynb` — filter to model scope, fix nulls semantically, validate, save final dataset.